<a href="https://colab.research.google.com/github/Adhiaris/UAS-FOR-MACHINE-LEARNING-Value-Predict/blob/main/Task_2_Value_Predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q optuna mlflow imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
import optuna
import mlflow
import mlflow.sklearn

import tensorflow as tf
from tensorflow import keras


In [ ]:
import pandas as pd

train = pd.read_csv('train_transaction.csv')
train = train.sample(frac=0.5, random_state=42).reset_index(drop=True)

In [ ]:
target = 'isFraud'
drop_cols = [c for c in ['TransactionID', 'TransactionDT'] if c in train.columns]

X = train.drop(columns=[target] + drop_cols)
y = train[target]

for col in X.select_dtypes(include='float64').columns:
    X[col] = X[col].astype('float32')
for col in X.select_dtypes(include='int64').columns:
    X[col] = X[col].astype('int32')

cat_cols = X.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X)
del X
X_train, X_val, y_train, y_val = train_test_split(X_imp, y, test_size=0.2, random_state=42, stratify=y)
del X_imp
X_train_res, y_train_res = SMOTE(random_state=42).fit_resample(X_train, y_train)
del X_train

## Machine Learning Model

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 100),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'random_state': 42,
        'n_jobs': -1
    }
    clf = RandomForestClassifier(**params)
    clf.fit(X_train_res, y_train_res)
    return roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)

In [ ]:
mlflow.set_experiment("midterm-fraud-detection")

with mlflow.start_run(run_name="RandomForest"):
    best_params = {**study.best_params, 'random_state': 42, 'n_jobs': -1}
    model = RandomForestClassifier(**best_params)
    model.fit(X_train_res, y_train_res)

    preds       = model.predict(X_val)
    preds_proba = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds_proba)

    mlflow.log_params(best_params)
    mlflow.log_metric("AUC-ROC", auc)
    mlflow.sklearn.log_model(model, "random_forest_model")

print(f"AUC-ROC: {auc:.4f}")
print(classification_report(y_val, preds))


In [ ]:
cm = confusion_matrix(y_val, preds)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Fraud', 'Fraud']).plot(cmap='Blues')
plt.title("Confusion Matrix (Random Forest)")
plt.tight_layout()
plt.show()

feat_df = pd.DataFrame({'feature': train.drop(columns=['isFraud','TransactionID','TransactionDT'], errors='ignore').columns, 'importance': model.feature_importances_})
feat_df = feat_df.sort_values('importance', ascending=False).head(15)
plt.figure(figsize=(9, 6))
plt.barh(feat_df['feature'][::-1], feat_df['importance'][::-1])
plt.xlabel("Importance")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()
